# Part 10 — Advanced RAG Architectures

*From a fixed retrieve-then-generate pipeline to a decision-making loop that grades what it got and corrects course.*

📖 Read the essay: https://www.mefby.com/essays/advanced-rag-architectures

🐍 Companion script: `corrective_rag.py`

## What you'll build

For nine parts we built a pipeline that runs the *same* path every time: retrieve once, stuff the chunks into a prompt, generate. This part is where that pipeline grows a brain. We add **control flow** — the branches and loops that let the system decide what to do next based on what it has seen.

We will hand-build **Corrective RAG (CRAG)**, the most concrete of the advanced architectures in the essay. CRAG inserts one disciplined check between *retrieve* and *generate*: a lightweight **retrieval evaluator** that grades the chunks, then acts on the grade *before* answering. Concretely you'll build:

- two tiny eyeball-able corpora: a policy index (our own store) and a simulated web fallback;
- an **embedder** with a transparent offline fallback (the Part 2 pattern);
- a **vector store + retriever** per source, so CRAG can route between them;
- a **retrieval evaluator** that grades chunks *relevant / ambiguous / irrelevant*;
- the two **corrective actions** — query reformulation and a web-search fallback;
- a grounded **generator** with an extractive offline stand-in;
- the full `corrective_rag()` control-flow loop, run over four worked cases including the flagship *irrelevant → web fallback*.

We'll also peek at the **Agentic RAG** ReAct loop that CRAG is one disciplined slice of.

> **Runs fully offline.** Every code cell executes with **no network and no API key** — numpy only. The real library / LLM path is shown as the headline (the code you'd actually ship); if a model or key is unavailable, a transparent deterministic stand-in keeps the cell runnable and prints a clear label when it kicks in.

## Setup

Two imports only: `re` for tokenizing and `numpy` for the vector math. `os` lets us check for an API key without ever requiring one. Nothing here touches the network.

In [ ]:
import os
import re
import numpy as np

print("numpy", np.__version__, "— ready (no network, no API key required)")

## Step 0 — Two tiny, eyeball-able corpora

CRAG needs two sources so it has somewhere to *correct to*.

**OUR INDEX** is the same store-policy corpus from Part 6: refunds, shipping, warranty. It deliberately holds **no product specs** — exactly the routing failure the flagship example is built around. When someone asks about earbud battery life, our own index simply cannot help.

**THE WEB** is the stubbed, *simulated* outside source CRAG falls back to. In production this is a real web-search API; offline we hard-code a couple of "pages" so the demo is honest about what a fallback would return without ever touching the network.

In [ ]:
OUR_INDEX = [
    "Refunds are accepted within 30 days of purchase, provided the item is unused and in its original packaging.",
    "To start a return, email support@example.com with your order number. Refunds are processed within five business days of us receiving the item.",
    "Standard shipping takes 3 to 5 business days. Express shipping arrives the next business day.",
    "Shipping fees are non-refundable, and items marked final sale cannot be returned or exchanged.",
    "All electronics include a one-year limited warranty covering manufacturing defects.",
]

# Simulated web 'pages' (the corrective fallback source). Offline + deterministic.
WEB_PAGES = [
    "The X1 wireless earbuds deliver up to 8 hours of battery life on a single charge, and up to 24 hours total with the charging case.",
    "The X1 wireless earbuds support Bluetooth 5.3 and are rated IPX4 for sweat and water resistance.",
    "Acme Corp was acquired by Globex in 2024; Globex now manufactures the X1 wireless earbuds.",
]

print(f"OUR index: {len(OUR_INDEX)} store-policy chunks (no product specs).")
print(f"Simulated web fallback: {len(WEB_PAGES)} pages.")

## Step 1 — Embeddings, with a transparent deterministic fallback

Everything downstream compares meaning as *closeness in a shared vector space* (Parts 2 and 3). The intended path is `sentence-transformers`, exactly as in Part 6 — a real model that learned to place similar meanings near each other.

This is the **headline**: the code you'd actually ship. We wrap the load in `try/except` so the notebook still runs with no model and no network — if the import or download fails, we drop to a transparent stand-in (next cell).

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+")


def _tokens(text):
    """Lowercase word/number tokens — the level the essay stays at."""
    return _TOKEN_RE.findall(text.lower())

### The offline stand-in: a deterministic hashing embedder

A real embedding model is a neural network we cannot reproduce without the weights. What we *can* do, with zero dependencies, is mimic the **interface**: any text → a fixed-length dense unit vector, deterministic and comparable in one space.

How: hash each token into a fixed-width vector and accumulate, then L2-normalize. Same text → same vector, always. Texts that share words land closer (shared tokens push the same dimensions the same way). It is crude — a hash has no idea "refund" and "reimbursement" are cousins — but it gives a real cosine score, enough to make the CRAG control flow concrete. `normalize=True` means a later dot product reads straight off as cosine similarity (the Part 3 trick).

In [ ]:
class _HashingEmbedder:
    """Deterministic, model-free stand-in for a sentence embedder (Part 2)."""

    def __init__(self, dim=256):
        self.dim = dim

    def encode(self, texts, normalize_embeddings=True):
        vecs = np.zeros((len(texts), self.dim), dtype=np.float64)
        for r, text in enumerate(texts):
            for tok in _tokens(text):
                # Stable hash -> bucket. Sign spreads collisions, like the
                # hashing trick. Deterministic across runs (no PYTHONHASHSEED).
                h = 0
                for ch in tok:
                    h = (h * 131 + ord(ch)) & 0xFFFFFFFF
                vecs[r, h % self.dim] += 1.0 if (h >> 1) & 1 else -1.0
        if normalize_embeddings:
            norms = np.linalg.norm(vecs, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            vecs = vecs / norms
        return vecs

Now the loader that prefers the real model and falls back transparently. It prints a clear label so you always know which path is live. **The real path is the headline; the stand-in just keeps the cell executable.**

In [ ]:
def _load_embedder():
    """Try the real model first; fall back transparently if unavailable."""
    try:
        from sentence_transformers import SentenceTransformer  # REAL path
        model = SentenceTransformer("all-MiniLM-L6-v2")         # 384 dims
        print("[embed] using sentence-transformers (all-MiniLM-L6-v2)")
        return model, True
    except Exception as exc:  # not installed / no weights / offline
        print(f"[embed] sentence-transformers unavailable ({type(exc).__name__}); "
              "falling back to deterministic hashing embedder")
        return _HashingEmbedder(), False


_EMBEDDER, _REAL_EMBEDDER = _load_embedder()


def embed(texts):
    return np.asarray(_EMBEDDER.encode(texts, normalize_embeddings=True))


# Quick sanity check: text in -> unit vectors out, in one shared space.
_v = embed(["refund policy", "earbud battery life"])
print(f"embed() returns shape {_v.shape}; row norms ≈ {np.round(np.linalg.norm(_v, axis=1), 3).tolist()}")

## Step 2 — A tiny vector store + retriever, one per source

A "vector store" is just chunks kept side by side with a matrix of their vectors (Part 4). `retrieve()` embeds the query with the **same** model, scores every chunk by cosine similarity (a dot product, since the vectors are unit length), and keeps the top-k (Part 3 + Part 4).

We build *one* store for `OUR_INDEX` and *one* for the simulated web, so CRAG can route between them.

In [ ]:
class VectorStore:
    def __init__(self, name, corpus):
        self.name = name
        self.chunks = [{"text": doc, "source": f"{name}_{i}"} for i, doc in enumerate(corpus)]
        self.vectors = embed([c["text"] for c in self.chunks])  # (n_chunks, dim)

    def retrieve(self, query, k=3):
        q = embed([query])[0]                       # same model as the chunks
        scores = self.vectors @ q                   # cosine sim (unit vectors)
        top = np.argsort(-scores)[:k]               # indices of k highest scores
        return [(self.chunks[i]["text"], float(scores[i])) for i in top]


OUR_STORE = VectorStore("policy", OUR_INDEX)
WEB_STORE = VectorStore("web", WEB_PAGES)


def retrieve(query, k=3):
    """CRAG searches our OWN index first (the essay's `retrieve(query)`)."""
    return OUR_STORE.retrieve(query, k=k)


# Peek: a policy question should pull a policy chunk to the top.
for text, score in retrieve("Are refunds accepted on unused items?"):
    print(f"  {score:+.3f}  {text[:60]}...")

## Step 3 — The retrieval evaluator: the one component CRAG adds

This is the heart of the part. A **retrieval evaluator** is a lightweight model or classifier whose only job is to grade the chunks retrieval returned. The grade is coarse — *relevant / ambiguous / irrelevant* — and the system acts on it **before** generating:

- **relevant** → the best chunk clearly answers the question, so answer now;
- **ambiguous** → some signal but not enough, so reformulate and retry our index;
- **irrelevant** → no chunk addresses the question, so our index can't help — fall back.

A naive pipeline generates from whatever retrieval hands it, even garbage. This one check between *retrieve* and *generate* is what lets the system catch a bad retrieval and fix it rather than confidently answering from chunks that don't contain the answer.

### The real path (headline, networked, never run offline)

In production the grader is typically a small LLM or classifier. Here is the shape with a hosted model. **Only this helper touches the network, and the demo never calls it offline** — it's the intended code, shown for reference.

In [ ]:
def grade_real(query, retrieved):
    """REAL path: ask a small LLM to grade the chunks (intended production code)."""
    from openai import OpenAI                       # SDK names move; check docs
    client = OpenAI()                               # reads OPENAI_API_KEY
    context = "\n".join(f"- {t}" for t, _ in retrieved)
    prompt = (
        "You grade whether retrieved context can answer a question.\n"
        "Reply with exactly one word: relevant, ambiguous, or irrelevant.\n\n"
        f"Question: {query}\n\nContext:\n{context}\n\nGrade:"
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    word = resp.choices[0].message.content.strip().lower()
    return word if word in {"relevant", "ambiguous", "irrelevant"} else "ambiguous"

print("grade_real defined (networked headline; not called offline).")

### The offline grader: cosine + content-word overlap

Thresholding a raw cosine alone is fragile — it shifts with whichever embedder is installed. So the offline grader blends cosine similarity with **content-word overlap**: the fraction of the query's *meaningful* words (stop-words removed) that the best chunk actually contains.

Overlap is what makes the flagship example honest. A product-spec question shares almost no content words with a policy-only index, so it grades **irrelevant** regardless of the embedder — exactly the routing failure CRAG is built to catch. First, the stop-word set the overlap metric ignores.

In [ ]:
# _STOPWORDS: grammatical words the overlap metric ignores when deciding
# whether a chunk addresses the query's CONTENT words.
_STOPWORDS = {
    "what", "is", "are", "the", "of", "a", "an", "for", "on", "in", "to", "how",
    "do", "does", "and", "my", "i", "there", "with", "your",
}


def _content_overlap(query, chunk_text):
    """Fraction of the query's content words that appear in the chunk."""
    q_words = {w for w in _tokens(query) if w not in _STOPWORDS}
    if not q_words:
        return 0.0
    c_words = set(_tokens(chunk_text))
    return len(q_words & c_words) / len(q_words)

Now blend the two signals into one `[0, 1]` score, and pick the chunk that best *answers* the query by that combined score. Ranking by the combined score (not raw cosine) keeps the evaluator and the generator consistent, and makes selection robust to which embedder is loaded.

In [ ]:
def _combined(query, text, cos):
    """Blend cosine similarity with content-word overlap into one [0,1] score."""
    return 0.5 * max(cos, 0.0) + 0.5 * _content_overlap(query, text)


def _best_chunk(query, retrieved):
    """The chunk that best ANSWERS the query, ranked by the combined score."""
    return max(retrieved, key=lambda r: _combined(query, r[0], r[1]))


def grade_score(query, retrieved):
    """Combined relevance signal for the best-answering chunk."""
    if not retrieved:
        return 0.0
    best_text, best_cos = _best_chunk(query, retrieved)
    return _combined(query, best_text, best_cos)

Finally, band the combined score into the three grades. The thresholds are chosen so the worked examples land on the right branch with **either** embedder (real MiniLM or the hashing fallback).

In [ ]:
RELEVANT_THRESHOLD = 0.50    # combined score at/above this -> trust the chunks
AMBIGUOUS_THRESHOLD = 0.20   # between the two bands -> some signal, reformulate
#                            below AMBIGUOUS_THRESHOLD -> irrelevant -> fall back


def grade_fallback(query, retrieved):
    """Deterministic, model-free grader: cosine similarity + content overlap."""
    score = grade_score(query, retrieved)
    if score >= RELEVANT_THRESHOLD:
        return "relevant"
    if score >= AMBIGUOUS_THRESHOLD:
        return "ambiguous"
    return "irrelevant"


# See the three bands in action on our own index:
for q in ["Are refunds accepted on unused items in original packaging?",
          "What is the battery life of the X1 wireless earbuds?"]:
    chunks = retrieve(q)
    print(f"{grade_fallback(q, chunks):<11} score={grade_score(q, chunks):.2f}  <- {q[:50]}")

### The evaluator object: real grader if available, else the stand-in

This mirrors the essay's `evaluator.grade(query, chunks)` call site. It uses the real LLM grader only if we have **both** a real embedder and an API key; otherwise it falls through to the transparent stand-in. Offline, that's always the stand-in.

In [ ]:
class Evaluator:
    """Mirrors the essay's `evaluator.grade(query, chunks)` call site."""

    def __init__(self, use_real):
        self.use_real = use_real

    def grade(self, query, retrieved):
        if self.use_real:
            try:
                return grade_real(query, retrieved)
            except Exception:
                pass  # fall through to the transparent stand-in
        return grade_fallback(query, retrieved)


# Use the real grader only if we have BOTH a real embedder and an API key.
evaluator = Evaluator(use_real=_REAL_EMBEDDER and bool(os.environ.get("OPENAI_API_KEY")))
print(f"evaluator using {'REAL LLM grader' if evaluator.use_real else 'offline deterministic grader'}")

## Step 4 — The two corrective actions

When the grade is bad, CRAG doesn't generate — it *acts*. There are two corrective moves:

- `reformulate()` lightly rewrites the query to try **our** index again (the *ambiguous* branch);
- `rewrite_for_web()` reshapes it for an outside source, and `web_search()` falls back to a **different** source (the *irrelevant* branch).

Reformulation strips low-signal **filler**. Keeping `_FILLER` separate from `_STOPWORDS` is deliberate: a chatty query can grade *ambiguous* on the first pass (the filler dilutes the cosine), and reformulating it down to its content words is what lets the retry succeed — exactly the *ambiguous → reformulate → retry* branch the essay describes.

In [ ]:
# _FILLER: a LARGER set of conversational chatter that reformulate() strips.
_FILLER = _STOPWORDS | {"can", "you", "tell", "me", "about", "please", "could", "would"}


def reformulate(query):
    """Ambiguous branch: tweak the query to retry our own index."""
    # A real system would call an LLM to paraphrase. Here we deterministically
    # drop low-signal filler so the next retrieval scores the content words.
    kept = [w for w in _tokens(query) if w not in _FILLER]
    reformed = " ".join(kept) if kept else query
    return reformed


def rewrite_for_web(query):
    """Irrelevant branch: reshape the query for an outside source."""
    return f"{reformulate(query)} specifications"


chatty = "Can you tell me about the warranty for electronics please?"
print(f"reformulate:    {chatty!r}")
print(f"            ->  {reformulate(chatty)!r}")
print(f"rewrite_for_web -> {rewrite_for_web('What is the battery life of the X1 wireless earbuds?')!r}")

### Web search: real API (headline) vs offline stand-in

In production `web_search()` is a real networked API call. Offline we query the simulated `WEB_STORE` instead — no network, deterministic. The real helper is shown as the intended code; only it would touch the network, and the demo never calls it.

In [ ]:
def web_search_real(query, k=3):
    """REAL path: a live web-search API. The only networked helper; unused offline."""
    import requests                                 # pragma: no cover
    resp = requests.get(
        "https://api.example-search.com/v1/search",  # placeholder endpoint
        params={"q": query, "k": k},
        timeout=10,
    )
    hits = resp.json()["results"]
    return [(h["snippet"], float(h.get("score", 0.0))) for h in hits]


def web_search(query, k=3):
    """Offline stand-in: query the simulated WEB_STORE instead of the network."""
    return WEB_STORE.retrieve(query, k=k)


# The web fallback DOES hold the earbud spec our policy index lacks:
for text, score in web_search("X1 wireless earbuds battery life specifications"):
    print(f"  {score:+.3f}  {text[:60]}...")

## Step 5 — Generation, with a grounded extractive fallback

The prompt template and grounding instruction are lifted straight from Part 6, so the model answers **only** from context and refuses otherwise. The intended path is a hosted LLM; offline we use an extractive stand-in that stitches the best retrieved chunk into a clearly-grounded reply — so the demo always produces sensible, source-backed output with no model.

In [ ]:
PROMPT_TEMPLATE = """Answer the question using ONLY the context below.
If the answer is not in the context, say \"I don't know based on the available sources.\"

Context:
{context}

Question: {question}
Answer:"""


def build_prompt(query, retrieved):
    context = "\n".join(f"- {text}" for text, _score in retrieved)
    return PROMPT_TEMPLATE.format(context=context, question=query)


print(build_prompt("Are refunds accepted?", retrieve("refund policy"))[:220], "...")

The real generator is one swappable hosted-LLM call (Part 6's `generate()`), shown as the headline. The offline stand-in quotes the best-answering chunk — the same ranking the grader uses — and reports a grounding score, so the answer is visibly tied to a source.

In [ ]:
def generate_real(prompt):
    """REAL path: one swappable hosted-LLM call (Part 6's generate())."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,                              # grounded, not creative
    )
    return resp.choices[0].message.content


def generate_fallback(query, retrieved):
    """Deterministic extractive generator: ground the answer in the chunks."""
    if not retrieved:
        return "I don't know based on the available sources."
    # Pick the best-ANSWERING chunk (same ranking as the grader) and quote it.
    best_text, best_cos = _best_chunk(query, retrieved)
    return (f"Based on the retrieved sources: {best_text} "
            f"(grounding score {_combined(query, best_text, best_cos):.2f})")


def generate(query, retrieved):
    """Build the prompt (real path) or extract offline. Always grounded."""
    if _REAL_EMBEDDER and os.environ.get("OPENAI_API_KEY"):
        try:
            return generate_real(build_prompt(query, retrieved))
        except Exception:
            pass
    return generate_fallback(query, retrieved)


print(generate("Are refunds accepted on unused items?", retrieve("refund unused original packaging")))

## Step 6 — The Corrective RAG control-flow loop

Now we assemble the pieces into the one function that *is* CRAG. This is a faithful, runnable expansion of the essay's `corrective_rag` sketch — same shape: **retrieve, grade, then either correct and retry, or generate.** The `verbose` tracing makes the branch each query takes visible.

Read the branches against the essay:

- `relevant` → good context, answer now;
- `irrelevant` → do *not* generate on bad context; rewrite for an outside source and fall back to web search (a different source), grading those too;
- `ambiguous` → reformulate the query and loop to try our index again;
- out of tries → refuse honestly (Part 6's grounding).

In [ ]:
def corrective_rag(query, max_tries=2, verbose=True):
    def log(msg):
        if verbose:
            print(msg)

    log(f'\nQUERY: "{query}"')
    for attempt in range(max_tries):
        log(f"  attempt {attempt + 1}/{max_tries}")
        chunks = retrieve(query)                       # search our own index first
        grade = evaluator.grade(query, chunks)         # relevant/ambiguous/irrelevant
        log(f'    retrieved query="{query}"  '
            f'grade_score={grade_score(query, chunks):.2f}  grade={grade}')

        if grade == "relevant":
            log("    -> RELEVANT: good context, answer now.")
            return generate(query, chunks)             # good context: answer now

        # bad context: do NOT generate on it. take a corrective action.
        if grade == "irrelevant":
            log("    -> IRRELEVANT: our index can't answer; fall back to web search.")
            query = rewrite_for_web(query)             # reformulate for outside source
            web_chunks = web_search(query)             # fall back to a DIFFERENT source
            web_grade = evaluator.grade(query, web_chunks)  # a real CRAG grades these too
            log(f'       web search query="{query}"  '
                f'grade_score={grade_score(query, web_chunks):.2f}  grade={web_grade}')
            if web_grade == "irrelevant":
                log("       -> web fallback also weak; refuse honestly.")
                return "I don't know based on the available sources."
            log("       -> web context usable, answer from it.")
            return generate(query, web_chunks)

        # ambiguous: tweak the query and loop to try our index again.
        log("    -> AMBIGUOUS: reformulate the query and retry our index.")
        query = reformulate(query)

    # ran out of tries without good context: refuse honestly (Part 6's grounding).
    log("  -> out of tries without good context; refuse honestly.")
    return "I don't know based on the available sources."

print("corrective_rag() ready.")

## Step 7 — The demo: four queries, four branches

Now we run the headline demo from `corrective_rag.py`, adapted to the offline path so you see real traced output. Four cases, each landing on a different branch.

**Case 1 — RELEVANT.** A policy question our index answers directly.

In [ ]:
ans = corrective_rag("Are refunds accepted on unused items in original packaging?")
print(f"ANSWER: {ans}")

**Case 2 — AMBIGUOUS → reformulate → retry.** A chatty, filler-heavy question whose first retrieval is only middling. CRAG does *not* generate on it; it reformulates down to the content words and tries our index again, which this time grades relevant.

In [ ]:
ans = corrective_rag("Can you tell me about the warranty for electronics please?")
print(f"ANSWER: {ans}")

**Case 3 — IRRELEVANT → web fallback.** The flagship example: a product-spec question against a policy-only corpus. Naive RAG would guess; CRAG grades the policy chunks irrelevant and routes to the web source, which *does* hold the spec.

In [ ]:
ans = corrective_rag("What is the battery life of the X1 wireless earbuds?")
print(f"ANSWER: {ans}")

**Case 4 — honest REFUSAL.** A question *neither* source can answer. After the corrective web fallback also comes back weak, CRAG refuses honestly rather than confidently guessing (Part 6's grounding).

In [ ]:
ans = corrective_rag("How do I reset my smart thermostat to factory settings?")
print(f"ANSWER: {ans}")

## Step 8 — Reference: the Agentic RAG ReAct loop CRAG is a slice of

CRAG above is *one disciplined slice* of the open-ended **Agentic RAG** loop. For contrast, here is the fuller shape the essay describes: an **agent** that runs a **reason → act → observe** loop (the **ReAct loop**), choosing among a couple of **tools** — retrieval is now just *one* tool, alongside a calculator — until it judges the task done.

This is reference only; the executable demo is the CRAG path above. We define it but only print its skeleton, so nothing open-ended actually runs.

In [ ]:
TOOLS = {
    "search_policy": lambda q: OUR_STORE.retrieve(q),   # retrieval is a tool
    "search_web":    lambda q: WEB_STORE.retrieve(q),   # routing target
    "calculator":    lambda expr: eval(expr),           # so it stops fumbling math
}


def react_agent(goal, llm, tools=TOOLS, max_steps=6):
    """Reference-only ReAct skeleton (NOT run here — needs a real `llm`)."""
    scratchpad = ""                       # the running reason/act/observe trace
    for _ in range(max_steps):
        # REASON: the model decides the next action from the goal + history.
        thought = llm(f"Goal: {goal}\n{scratchpad}\n"
                      f"Available tools: {list(tools)}\n"
                      "Think, then emit: ACTION <tool> <input>  OR  FINISH <answer>")
        if thought.startswith("FINISH"):
            return thought[len('FINISH'):].strip()       # task judged done
        # ACT: parse and call the chosen tool (number/order chosen at run time).
        _, tool, arg = thought.split(maxsplit=2)
        observation = tools[tool](arg)
        # OBSERVE: feed the result back in and loop. Multi-hop when one
        # observation (e.g. 'Globex acquired Acme') feeds the next retrieval.
        scratchpad += f"\nACTION {tool} {arg}\nOBSERVATION {observation}"
    return "Stopped: agent did not converge within step budget."


print("react_agent defined (reference only). Tools available:", list(TOOLS))
print("The catch: agents are powerful but slower (many model calls), costlier,")
print("less predictable, and harder to debug. CRAG buys most of the win — catching")
print("a bad retrieval — with one evaluation step and one branch.")

## What you learned

- The leap of this part is from a **fixed pipeline** (retrieve, then generate, the same way every time) to a **decision-making loop** with **control flow** — deciding whether the context is good, and correcting when it isn't.
- **Corrective RAG (CRAG)** adds exactly one component: a **retrieval evaluator** that grades chunks *relevant / ambiguous / irrelevant* and acts on the grade *before* generating. We built it as a transparent cosine-plus-content-overlap grader so the branch each query takes is honest with any embedder.
- You watched the four branches fire: **relevant** answers now, **ambiguous** reformulates and retries our index, **irrelevant** rewrites and falls back to a different source (the flagship web fallback), and an exhausted search **refuses honestly** rather than guessing.
- CRAG is one disciplined slice of the open-ended **Agentic RAG** ReAct loop — powerful, but slower, costlier, and harder to debug. The honest default stays well-tuned simple RAG (Parts 6–9); reach for one advanced architecture only when measurement, not excitement, proves you need it.
- Everything ran **fully offline**: the real model / LLM / web paths are the headline code, with transparent deterministic stand-ins so every cell executes with no network and no API key.

### Next

We added control flow and architectures freely — but how do we know any of it *helped*? **Part 11 — Evaluating RAG** measures it: faithfulness, answer relevance, context precision and recall, and the frameworks that quantify them. (Previous: **Part 9 — Advanced Retrieval Patterns**.)